# Plant Disease Detection Pipeline

This notebook trains a symptom detection model on plant leaves and integrates a decision tree to predict the final disease.

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
# Define symptoms
symptoms_list = ['spots', 'yellowing', 'wilting', 'powder', 'leaf_curl', 'stem_damage', 'mold_growth']

# Symptom Mapping Heuristics based on disease names
def get_symptoms(class_name):
    name = class_name.lower()
    symptoms = [0, 0, 0, 0, 0, 0, 0]
    
    if 'healthy' in name:
        return symptoms
        
    if any(x in name for x in ['spot', 'scab', 'blight', 'rust', 'measles']):
        symptoms[0] = 1 # spots
    if any(x in name for x in ['yellow', 'mosaic', 'greening']):
        symptoms[1] = 1 # yellowing
    if any(x in name for x in ['wilt', 'blight', 'scorch']):
        symptoms[2] = 1 # wilting
    if any(x in name for x in ['mildew', 'mold']):
        symptoms[3] = 1 # powder
        symptoms[6] = 1 # mold_growth
    if 'curl' in name:
        symptoms[4] = 1 # leaf_curl
    if any(x in name for x in ['rot', 'canker']):
        symptoms[5] = 1 # stem_damage
        
    return symptoms

class SymptomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        for class_name in os.listdir(root_dir):
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                symptoms = get_symptoms(class_name)
                for img_name in os.listdir(class_dir):
                    if img_name.endswith(('.jpg', '.JPG', '.png', '.PNG')):
                        self.image_paths.append(os.path.join(class_dir, img_name))
                        self.labels.append(symptoms)
                        
    def __len__(self):
        return len(self.image_paths)
        
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Create dataset and dataloaders
data_dir = 'PlantVillage-Dataset-master/raw/color'
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = SymptomDataset(data_dir, transform=transform)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"Total images: {len(dataset)}, Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Total images: 54304, Train: 43443, Val: 10861


In [4]:
# Build and Train the Model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 7) # 7 symptoms
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10 # Increased epochs

print("Starting training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        
    epoch_loss = running_loss / len(train_dataset)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')

torch.save(model.state_dict(), 'final_plant_model_v2.pth')
print("Model saved to final_plant_model_v2.pth")

Starting training...


KeyboardInterrupt: 

In [ ]:
# ── Import everything from the shared module ──────────────────────────────────
import os, torch
from Plant_Disease_Detection import (
    build_model, SYMPTOM_NAMES, device, val_transform,
    PlantSymptomDataset, get_coords, predict_symptoms_v3,
    hybrid_diagnosis_final, plot_symptom_probs, IN_COLAB, load_and_train_tabular_model
)

# ── Load / initialise model ────────────────────────────────────────────────────
_model = build_model(num_classes=len(SYMPTOM_NAMES)).to(device)

for _ckpt in ('final_plant_model_v2.pth',
               'models/final_plant_model_v2.pth',):
    if os.path.exists(_ckpt):
        _model.load_state_dict(torch.load(_ckpt, map_location=device, weights_only=True))
        print(f"Loaded model weights from: {_ckpt}")
        break
else:
    print("No saved weights found – using untrained model.")

# Also initialise tabular model
load_and_train_tabular_model()

# ── Inference function ─────────────────────────────────────────────────────────
def final_production_inference(model=_model):
    print("--- Plant Disease Diagnosis System ---")

    if IN_COLAB:
        from google.colab import files as _files
        uploaded = _files.upload()
        if not uploaded:
            return
        img_path = list(uploaded.keys())[0]
    else:
        img_path = input("Enter the path to the plant photo: ").strip('"\' ')
        if not os.path.exists(img_path):
            print("File not found!")
            return

    loc = input("Enter location (e.g. 'asansol, india'): ")
    lat, lon = get_coords(loc)

    # 1. Vision Analysis
    results, vector = predict_symptoms_v3(img_path, model, val_transform, device, threshold=0.4)

    # 2. Hybrid Report
    hybrid_diagnosis_final(img_path, lat, lon, model, val_transform, device)

    # 3. Visual Insight
    plot_symptom_probs(model,
                       PlantSymptomDataset([{'path': img_path,
                                             'disease': 'User Input',
                                             'symptoms': [0]*7}],
                                           transform=val_transform),
                       0, device)

    if IN_COLAB and os.path.exists(img_path):
        os.remove(img_path)

final_production_inference()


Loaded EfficientNet-B0 weights from: final_plant_model.pth


RuntimeError: Error(s) in loading state_dict for EfficientNet:
	Missing key(s) in state_dict: "conv_stem.weight", "blocks.0.0.conv_dw.weight", "blocks.0.0.bn1.weight", "blocks.0.0.bn1.bias", "blocks.0.0.bn1.running_mean", "blocks.0.0.bn1.running_var", "blocks.0.0.se.conv_reduce.weight", "blocks.0.0.se.conv_reduce.bias", "blocks.0.0.se.conv_expand.weight", "blocks.0.0.se.conv_expand.bias", "blocks.0.0.conv_pw.weight", "blocks.0.0.bn2.weight", "blocks.0.0.bn2.bias", "blocks.0.0.bn2.running_mean", "blocks.0.0.bn2.running_var", "blocks.1.0.conv_pw.weight", "blocks.1.0.bn1.weight", "blocks.1.0.bn1.bias", "blocks.1.0.bn1.running_mean", "blocks.1.0.bn1.running_var", "blocks.1.0.conv_dw.weight", "blocks.1.0.bn2.weight", "blocks.1.0.bn2.bias", "blocks.1.0.bn2.running_mean", "blocks.1.0.bn2.running_var", "blocks.1.0.se.conv_reduce.weight", "blocks.1.0.se.conv_reduce.bias", "blocks.1.0.se.conv_expand.weight", "blocks.1.0.se.conv_expand.bias", "blocks.1.0.conv_pwl.weight", "blocks.1.0.bn3.weight",